In [2]:
from pathlib import Path
import pandas as pd
import pyrosetta
from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.protocols.backrub import BackrubMover
pyrosetta.init("-mute all")

PDB_PATH = Path("/mnt/c/Users/kevin/Downloads/RYR2_PDB_STRUCTURES/PDB_STRUCTURES/5goa_K167A.pdb") #change naame for models
OUTDIR = Path("/mnt/c/Users/kevin/Downloads/K167A open backrub models") #change name for models
OUTDIR.mkdir(parents=True, exist_ok=True)

CHAIN = "A"
PDB_FLEX_START = 165 #residue numbers to experiment with
PDB_FLEX_END = 174

NSTRUCT = 20 #models generated
BACKRUB_TRIALS = 100 #how many do we want?

pose0 = pose_from_pdb(str(PDB_PATH))
scorefxn = pyrosetta.create_score_function("ref2015")
original_score = scorefxn(pose0)
print(f"Original score: {original_score:.3f} REU")
pdb_info = pose0.pdb_info()

FLEX_START = pdb_info.pdb2pose(CHAIN, PDB_FLEX_START) #convert pose to pdb pose
FLEX_END = pdb_info.pdb2pose(CHAIN, PDB_FLEX_END)
movemap = MoveMap()
movemap.set_bb_true_range(FLEX_START, FLEX_END)

backrub = BackrubMover()
backrub.set_movemap(movemap)
backrub.set_min_atoms(3) #change accordingly
backrub.set_max_atoms(12)

results = []

for i in range(1, NSTRUCT + 1):
    pose = pose0.clone()

    for trial in range(BACKRUB_TRIALS):
        backrub.apply(pose)

    model_score = scorefxn(pose)
    delta = model_score - original_score

    outfile = OUTDIR / f"5goa_K167A_backrub_{i:02d}.pdb" #change name for models
    pose.dump_pdb(str(outfile))

    print(f"Model {i:02d} | "f"Score: {model_score:.3f} REU | "f"Delta: {delta:.3f} REU")
    results.append({"model_number": i, "final_REU": model_score, "delta_from_original_REU": delta})

df = pd.DataFrame(results)
csv_out = OUTDIR / "K167A_open_backrub_scores.csv" #change name for models
df.to_csv(csv_out, index=False)

display(df)

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.Release.python312.ubuntu 2025.37+release.df75a9c48e763e52a7aa3f5dfba077f4da88dbf5 2025-09-03T12:23:30] retrieved from: http://www.pyrosetta.org
Original score: 423.062 REU
Model 01 | Score: 2330.001 REU | Delta: 1906.939 REU
Model 02 | Score: 555.125 REU | Delta: 132.063 REU
Model 03 | Score: 926.570 REU |

,model_number,final_REU,delta_from_original_REU
0,1,2330.001373,1906.939162
1,2,555.125469,132.063258
2,3,926.569990,503.507779
3,4,466.658298,43.596087
4,5,931.860367,508.798156
5,6,1952.653257,1529.591046
6,7,2459.272421,2036.210210
7,8,633.918552,210.856341
8,9,951.633975,528.571764
9,10,1104.568042,681.505831
